# Phase 2 : 피처 엔지니어링 파이프라인

## 1. Parquet 파일 로드

In [1]:
import polars as pl
import numpy as np
import pandas as pd

# Parquet 파일 읽기
df = pl.read_parquet("./데이터/processed_data.parquet")

print(f"원래 데이터 전체 행 수: {df.height:,} 건, 유저 수: {df['userId'].n_unique():,} 명")

# 🚨 [핵심 수정] LSTM 모델의 데이터 누수(AUC 1.0)를 유발하는 치트키 페이지 제거
# 이탈을 직접적으로 유출하는 결정적 페이지들을 로그 시퀀스에서 제외합니다.
leakage_pages = [
    'Cancel', 
    'Cancellation Confirmation', 
    'Submit Downgrade',
    'Downgrade',          # 추가
    'Submit Upgrade',     # 추가
    'Upgrade'             # 추가
]
df = df.filter(~pl.col("page").is_in(leakage_pages))

print(f"누수 페이지 필터링 후 행 수: {df.height:,} 건 (유저 수: {df['userId'].n_unique():,} 명)")
print(df.head(3))

원래 데이터 전체 행 수: 26,259,199 건, 유저 수: 22,278 명
누수 페이지 필터링 후 행 수: 25,992,817 건 (유저 수: 22,278 명)
shape: (3, 21)
┌────────┬────────┬───────────┬───────────┬───┬────────┬────────────────┬───────────────┬──────────┐
│ status ┆ gender ┆ length    ┆ firstName ┆ … ┆ method ┆ datetime       ┆ reg_datetime  ┆ is_churn │
│ ---    ┆ ---    ┆ ---       ┆ ---       ┆   ┆ ---    ┆ ---            ┆ ---           ┆ ---      │
│ i64    ┆ str    ┆ f64       ┆ str       ┆   ┆ str    ┆ datetime[μs]   ┆ datetime[μs]  ┆ i32      │
╞════════╪════════╪═══════════╪═══════════╪═══╪════════╪════════════════╪═══════════════╪══════════╡
│ 200    ┆ M      ┆ null      ┆ Camren    ┆ … ┆ GET    ┆ 2018-10-02     ┆ 2018-07-10    ┆ 1        │
│        ┆        ┆           ┆           ┆   ┆        ┆ 08:59:29       ┆ 09:30:08      ┆          │
│ 200    ┆ M      ┆ 251.97669 ┆ Camren    ┆ … ┆ PUT    ┆ 2018-10-02     ┆ 2018-07-10    ┆ 1        │
│        ┆        ┆           ┆           ┆   ┆        ┆ 09:02:44       ┆ 09:30:08   

## 2. 유저별 정형 피처(Tabular Features) 생성
### 가입 기간(Tenure), 총 재생 곡 수, 유료/무료 여부, 총 세션 수 등의 정적/누적 통계를 추출

In [2]:
# 1. 💡 [핵심 수정] 전체 데이터를 시간순(ts)으로 먼저 정렬한 뒤 유저별로 그룹화합니다.
# 이렇게 하면 .last()를 사용할 때 가장 마지막(최근)에 발생한 데이터가 정확히 선택됩니다.
user_tabular_features = (
    df.sort("ts")
    .group_by("userId")
    .agg([
        pl.col("is_churn").first().alias("is_churn"),
        pl.col("gender").first().alias("gender"),
        pl.col("level").last().alias("level"),
        
        # 원본 타임스탬프 및 세션 관련 (나중에 tenure_days 계산 등에 활용)
        pl.col("ts").max().alias("max_ts"),
        pl.col("registration").first().alias("registration"),
        
        # 기본 누적 통계
        pl.len().alias("total_interactions"),
        (pl.col("page") == "NextSong").sum().alias("total_songs_played"),
        (pl.col("page") == "Thumbs Up").sum().alias("total_thumbs_up"),
        (pl.col("page") == "Thumbs Down").sum().alias("total_thumbs_down"),
        (pl.col("page") == "Add to Playlist").sum().alias("total_playlist_adds"),
        
        # 서비스 불편도 및 광고 피로도 피처
        (pl.col("page") == "Error").sum().alias("total_errors"),
        (pl.col("page") == "Help").sum().alias("total_help"),
        (pl.col("page") == "Roll Over Ad").sum().alias("total_ads"),
        
        # 🚀 [추가] 유저의 가장 마지막(최근) 행동 페이지 포착 (시간순 정렬되어 있으므로 .last()가 최신값)
        pl.col("page").last().alias("last_page"),
        
        pl.col("sessionId").n_unique().alias("total_sessions"),
        pl.col("length").sum().alias("total_listening_time_sec"),
    ])
)

# 2. 곧바로 Pandas DataFrame으로 변환
pdf = user_tabular_features.to_pandas()

# 3. 누수 없는 상대 비율 및 파생 피처 계산
pdf["tenure_days"] = (pdf["max_ts"] - pdf["registration"]) / (
    1000 * 60 * 60 * 24
)
pdf["thumbs_down_ratio"] = pdf["total_thumbs_down"] / (
    pdf["total_thumbs_up"] + 1.0
)
pdf["songs_per_session"] = pdf["total_songs_played"] / (
    pdf["total_sessions"] + 1.0
)
pdf["activity_density"] = pdf["total_interactions"] / (pdf["tenure_days"] + 1.0)

# 추가 고성능 피처
pdf["error_ratio"] = pdf["total_errors"] / (pdf["total_interactions"] + 1.0)
pdf["help_ratio"] = pdf["total_help"] / (pdf["total_interactions"] + 1.0)
pdf["ad_ratio"] = pdf["total_ads"] / (pdf["total_interactions"] + 1.0)
pdf["listening_time_per_session"] = pdf["total_listening_time_sec"] / (
    pdf["total_sessions"] + 1.0
)

# 🚨 절대 타임스탬프 원본 컬럼은 모델 학습 누수를 막기 위해 즉시 제거
pdf = pdf.drop(columns=["max_ts", "registration"], errors="ignore")

# 4. 수치형 피처 타입 강제 변환 및 결측치 처리
numeric_cols = [
    "tenure_days",
    "total_interactions",
    "total_songs_played",
    "total_thumbs_up",
    "total_thumbs_down",
    "total_playlist_adds",
    "total_sessions",
    "total_listening_time_sec",
    "thumbs_down_ratio",
    "songs_per_session",
    "activity_density",
    "error_ratio",
    "help_ratio",
    "ad_ratio",
    "listening_time_per_session",
]

for col in numeric_cols:
  if col in pdf.columns:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce").fillna(0.0)

print("최근성 및 불만족 피처가 포함된 정형 피처 생성 완료!")

최근성 및 불만족 피처가 포함된 정형 피처 생성 완료!


## 3. 유저별 행동 시퀀스(Sequence) 데이터 추출
### 딥러닝 모델 입력을 위해 유저별로 발생한 page 이동 기록을 시간 순서대로 정렬하여 리스트 형태로 수집

In [3]:
# 유저별로 최근 최대 N개의 이벤트 시퀀스 추출 (예: 최근 50개 행동)
MAX_SEQ_LEN = 50

user_sequences = (
    df.sort(["userId", "ts"])
    .group_by("userId")
    .agg([
        pl.col("page").tail(MAX_SEQ_LEN).alias("page_sequence"),
        pl.col("itemInSession").tail(MAX_SEQ_LEN).alias("session_item_sequence")
    ])
)

# 정형 피처와 시퀀스 피처를 유저 ID 기준 병합
user_master_df = user_tabular_features.join(user_sequences, on="userId")

print(f"최종 마스터 유저 테이블 크기: {user_master_df.shape}")
print(user_master_df.select(["userId", "is_churn", "page_sequence"]).head(3))

최종 마스터 유저 테이블 크기: (22278, 19)
shape: (3, 3)
┌─────────┬──────────┬─────────────────────────────────┐
│ userId  ┆ is_churn ┆ page_sequence                   │
│ ---     ┆ ---      ┆ ---                             │
│ str     ┆ i32      ┆ list[str]                       │
╞═════════╪══════════╪═════════════════════════════════╡
│ 1000025 ┆ 1        ┆ ["NextSong", "Home", … "NextSo… │
│ 1000035 ┆ 0        ┆ ["Thumbs Up", "NextSong", … "T… │
│ 1000083 ┆ 1        ┆ ["NextSong", "NextSong", … "Ne… │
└─────────┴──────────┴─────────────────────────────────┘


## 4. 시퀀스 토큰화(Tokenization) 및 정수 인코딩
### 문자열 페이지 이름을 숫자로 변환하고, 딥러닝 입력을 위한 고정 길이 텐서 형태로 가공

In [4]:
import pandas as pd # 패딩 및 매핑 처리를 위해 Pandas 병행 활용

# Polars DataFrame을 Pandas로 변환하여 파이썬 네이티브 연산 적용
pdf = user_master_df.to_pandas()

# 1. 고유 페이지(Page) 어휘 사전(Vocabulary) 구축
unique_pages = pdf['page_sequence'].explode().dropna().unique()
page_to_idx = {page: idx + 1 for idx, page in enumerate(unique_pages)}  # 0번은 Padding용으로 비워둠
page_to_idx['<PAD>'] = 0

print(f"고유 페이지 종류: {len(page_to_idx)-1} 개")

# 2. 정수 인코딩 및 고정 길이 패딩(Padding) 함수 정의
def encode_and_pad(seq, max_len=MAX_SEQ_LEN):
    encoded = [page_to_idx.get(p, 0) for p in seq]
    
    if len(encoded) == 0:
        encoded = [0] * max_len
    elif len(encoded) < max_len:
        # 🚨 [핵심 수정] 0으로 채우는 대신, 기존 시퀀스를 반복(Loop)하여 정확히 max_len을 채움
        # 이를 통해 활동량 길이로 인한 데이터 누수(AUC 1.0)를 원천 차단합니다.
        repeated = []
        while len(repeated) < max_len:
            repeated.extend(encoded)
        encoded = repeated[:max_len]
    else:
        # 길이가 초과하면 최근 N개만 유지
        encoded = encoded[-max_len:]
        
    return encoded

pdf['padded_sequence'] = pdf['page_sequence'].apply(lambda x: encode_and_pad(x))

# 3. 넘파이 배열(Numpy Array)로 변환
X_seq = np.array(pdf['padded_sequence'].tolist())
y = pdf['is_churn'].values

print(f"딥러닝 시퀀스 입력 텐서(X_seq) 크기: {X_seq.shape}")
print(f"타겟 라벨(y) 크기: {y.shape}, 이탈 유저 비율: {y.mean():.4f}")

고유 페이지 종류: 14 개
딥러닝 시퀀스 입력 텐서(X_seq) 크기: (22278, 50)
타겟 라벨(y) 크기: (22278,), 이탈 유저 비율: 0.2246


## 5. 피처 데이터셋 저장
### GBDT용 정형 데이터는 Parquet으로, 딥러닝용 시퀀스 텐서는 Numpy 파일(.npy)로 저장

In [5]:
# 1. 정형 피처 테이블 저장 (시퀀스 리스트 컬럼 제외)
pdf.drop(columns=['page_sequence', 'padded_sequence', 'max_ts', 'registration'], errors='ignore').to_parquet("./데이터/ml_tabular_features.parquet")

# 2. 딥러닝 시퀀스 텐서 및 타겟 라벨 저장
np.save("./데이터/dl_sequence_tensor.npy", X_seq)
np.save("./데이터/churn_labels.npy", y)

print("Phase 2 피처 엔지니어링 완료!")
print("- ML용 정형 피처: 'ml_tabular_features.parquet'")
print("- DL용 시퀀스 텐서: 'dl_sequence_tensor.npy'")
print("- 타겟 라벨: 'churn_labels.npy'")

Phase 2 피처 엔지니어링 완료!
- ML용 정형 피처: 'ml_tabular_features.parquet'
- DL용 시퀀스 텐서: 'dl_sequence_tensor.npy'
- 타겟 라벨: 'churn_labels.npy'
